<a href="https://colab.research.google.com/github/lgelara/prismaFRL/blob/main/Lucia_prisma_paso3_analisis_parte3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 PRISMA — Paso 3: Análisis Bibliométrico

Este cuaderno genera análisis bibliométricos sobre los artículos elegibles del Paso 2.

| Sección | Análisis |
|---------|----------|
| A | WordCloud de títulos y abstracts |
| B | Tendencias de publicación por año + regresión |
| C | Top autores y revistas |
| D | Co-ocurrencia de keywords |
| E | Análisis automático de columnas disponibles |

**Instrucciones:** Ejecuta las celdas en orden. La libreta detecta automáticamente las columnas disponibles en tu archivo.

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 1 · Dependencias
# ─────────────────────────────────────────────────────────────
!pip install wordcloud plotly pandas numpy scipy matplotlib kaleido -q
print('✓ Dependencias instaladas')

✓ Dependencias instaladas


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 2 · CONFIGURACIÓN (ACTUALIZADA)
# ─────────────────────────────────────────────────────────────

# ── Idioma de las gráficas (Configurado a Inglés) ─────────────
LANG = 'EN'

# ── Traducciones para los elementos visuales ──────────────────
TEXTS = {
    'EN': {
        'count': 'Count',
        'year': 'Year',
        'author': 'Authors',
        'journal': 'Journals/Sources',
        'country': 'Countries',
        'keywords': 'Keywords',
        'co_occurrence': 'Keyword Co-occurrence',
        'geo_dist': 'Geographic Distribution of Publications',
        'top_n': 'Top {n} {label}',
        'publications': 'Number of Publications'
    }
}

# ── Stopwords extra (palabras a ignorar en wordclouds) ────────
STOPWORDS_EXTRA = [
    'study', 'analysis', 'using', 'based', 'review',
    'research', 'paper', 'approach', 'method', 'results',
    'system', 'process', 'development', 'Vehicle', 'Health', 'smart cities'
]

# ── Configuración Visual ──────────────────────────────────────
WC_COLORMAP_TITULO   = 'Blues'
WC_COLORMAP_ABSTRACT = 'Greens'
GEO_COLORMAP         = 'YlGnBu'  # Mapa de calor para países
WC_MAX_WORDS         = 80
TOP_N                = 15
TOP_COOC             = 20

# ── Carpeta de salida para PNG ────────────────────────────────
CARPETA_PNG = 'bibliometric_figures'

# ─────────────────────────────────────────────────────────────
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

os.makedirs(CARPETA_PNG, exist_ok=True)
print(f'Configuration ready. Language: {LANG}. PNG Figures → ./{CARPETA_PNG}/')

# ── Función sugerida para la gráfica geográfica ───────────────
def plot_geographic_distribution(df_countries, top_n=TOP_N):
    """
    Asumiendo que tienes un DataFrame con la columna 'Country' y su conteo.
    """
    plt.figure(figsize=(10, 6))
    data = df_countries.head(top_n)

    sns.barplot(
        x=data.values,
        y=data.index,
        palette=GEO_COLORMAP
    )

    plt.title(TEXTS[LANG]['geo_dist'], fontsize=14)
    plt.xlabel(TEXTS[LANG]['publications'])
    plt.ylabel(TEXTS[LANG]['country'])

    plt.tight_layout()
    plt.savefig(f'{CARPETA_PNG}/geographic_distribution.png', dpi=300)
    plt.show()

Configuration ready. Language: EN. PNG Figures → ./bibliometric_figures/


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 3 · Cargar prisma_elegibles.csv
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
from IPython.display import display, HTML
from google.colab import files

print('Sube el archivo prisma_elegibles.csv:')
subido = files.upload()
nombre_csv = list(subido.keys())[0]
with open(nombre_csv, 'wb') as f:
    f.write(subido[nombre_csv])

df = pd.read_csv(nombre_csv, encoding='utf-8', on_bad_lines='skip')
print(f'\n✓ {len(df)} registros cargados')
print(f'   {len(df.columns)} columnas disponibles:')

# Detectar automáticamente columnas clave
def detectar(df, pistas):
    for p in pistas:
        hits = [c for c in df.columns if p.lower() in c.lower()]
        if hits:
            return hits[0]
    return None

COL_TITULO   = detectar(df, ['title','titulo','titulo_original'])
COL_ABSTRACT = detectar(df, ['abstract','resumen'])
COL_AÑO      = detectar(df, ['year','año','date','publication_date'])
COL_AUTORES  = detectar(df, ['author','autor','authors'])
COL_REVISTA  = detectar(df, ['journal','revista','source','fuente','Publication Title'])
COL_KEYWORDS = detectar(df, ['keyword','palabras','mesh','index keyword','author keyword'])
COL_DOI      = detectar(df, ['doi'])
COL_CITAS    = detectar(df, ['cited','citas','citation','times cited'])
COL_PAIS     = detectar(df, ['country','pais','país'])
COL_AFIL     = detectar(df, ['affiliation','afiliacion','institution'])

detecciones = {
    'Título':    COL_TITULO,
    'Abstract':  COL_ABSTRACT,
    'Año':       COL_AÑO,
    'Autores':   COL_AUTORES,
    'Revista':   COL_REVISTA,
    'Keywords':  COL_KEYWORDS,
    'DOI':       COL_DOI,
    'Citas':     COL_CITAS,
    'País':      COL_PAIS,
    'Afiliación':COL_AFIL,
}

print('\nColumnas detectadas automáticamente:')
for campo, col in detecciones.items():
    estado = f'✓  {col}' if col else '—  no encontrada'
    print(f'  {campo:12}: {estado}')

print('\nTodas las columnas del archivo:')
for c in df.columns:
    n_llenos = df[c].notna().sum()
    print(f'  {c:40} {n_llenos}/{len(df)} ({n_llenos/len(df)*100:.0f}%)')

Sube el archivo prisma_elegibles.csv:


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 4 · Funciones auxiliares  (no modificar)
# ─────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from wordcloud import WordCloud, STOPWORDS
from scipy import stats
from itertools import combinations
from collections import Counter

# Stopwords combinadas
SW = STOPWORDS.union(set(w.lower() for w in STOPWORDS_EXTRA))

def limpiar_texto(serie):
    """Une textos de una columna en un solo string limpio."""
    textos = serie.dropna().astype(str)
    texto  = ' '.join(textos)
    texto  = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', texto)
    texto  = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def guardar_fig_plotly(fig, nombre):
    ruta_html = f'{CARPETA_PNG}/{nombre}.html'
    ruta_png  = f'{CARPETA_PNG}/{nombre}.png'
    fig.write_html(ruta_html)
    try:
        fig.write_image(ruta_png, scale=2, width=1200, height=700)
        print(f'  PNG  → {ruta_png}')
    except Exception:
        print(f'  PNG  → no disponible (kaleido); usa HTML')
    print(f'  HTML → {ruta_html}')

def guardar_fig_mpl(fig, nombre):
    ruta = f'{CARPETA_PNG}/{nombre}.png'
    fig.savefig(ruta, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'  PNG  → {ruta}')

print('✓ Helpers cargados')

✓ Helpers cargados


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 5 · SECTION A — WordCloud & Word Frequencies (TITLES)
# ─────────────────────────────────────────────────────────────

from collections import Counter

if not COL_TITULO:
    print('⚠ Title column not detected. Please adjust COL_TITULO manually.')
else:
    # 1. Preparar texto
    texto_titulos = limpiar_texto(df[COL_TITULO])

    # 2. Generar WordCloud
    wc = WordCloud(
        width=1400, height=700,
        background_color='white',
        stopwords=SW,
        max_words=WC_MAX_WORDS,
        colormap=WC_COLORMAP_TITULO,
        collocations=False, # Cambiado a False para conteo exacto de palabras individuales
        prefer_horizontal=0.85,
    ).generate(texto_titulos)

    # 3. Visualización de la Imagen
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('WordCloud — Article Titles', fontsize=16, pad=12, fontweight='bold')
    plt.tight_layout()
    guardar_fig_mpl(fig, 'wordcloud_titles')
    plt.show()

    # 4. Cálculo y despliegue de frecuencias numéricas
    # Filtramos el texto para contar palabras que no sean stopwords
    palabras_lista = [p for p in texto_titulos.split() if p.lower() not in SW]
    conteo_frecuencias = Counter(palabras_lista)

    # Convertir a DataFrame para una vista limpia
    df_frecuencias = pd.DataFrame(
        conteo_frecuencias.most_common(TOP_N),
        columns=['Word', 'Frequency']
    )

    print(f"\n✓ WordCloud generated.")
    print(f"--- Top {TOP_N} Most Frequent Words in Titles ---")
    print(df_frecuencias.to_string(index=False))

  PNG  → bibliometric_figures/wordcloud_titles.png

✓ WordCloud generated.
--- Top 15 Most Frequent Words in Titles ---
         Word  Frequency
     Learning         65
Reinforcement         62
    Federated         57
  Transformer         19
     Networks         15
      Enabled         11
     Resource         10
       Energy         10
          IoT          9
         Edge          9
          UAV          9
      Control          7
      Systems          7
    Computing          7
   Allocation          7


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 6 · SECTION A — WordCloud & Word Frequencies (ABSTRACTS)
# ─────────────────────────────────────────────────────────────

from collections import Counter

if not COL_ABSTRACT:
    print('⚠ Abstract column not detected.')
else:
    # 1. Clean and prepare text
    texto_abs = limpiar_texto(df[COL_ABSTRACT])

    # 2. Generate WordCloud
    wc_abs = WordCloud(
        width=1400, height=700,
        background_color='white',
        stopwords=SW,
        max_words=WC_MAX_WORDS,
        colormap=WC_COLORMAP_ABSTRACT,
        collocations=False, # Set to False for consistency with the frequency count
        prefer_horizontal=0.85,
    ).generate(texto_abs)

    # 3. Plotting the Image
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.imshow(wc_abs, interpolation='bilinear')
    ax.axis('off')
    ax.set_title('WordCloud — Article Abstracts', fontsize=16, pad=12, fontweight='bold')
    plt.tight_layout()
    guardar_fig_mpl(fig, 'wordcloud_abstracts')
    plt.show()

    # 4. Calculation of Word Frequencies
    # Filter text to count words not in stopwords (SW)
    abs_words_list = [word for word in texto_abs.split() if word.lower() not in SW]
    abs_frequency_count = Counter(abs_words_list)

    # Create a DataFrame for the Top N words
    df_abs_freq = pd.DataFrame(
        abs_frequency_count.most_common(TOP_N),
        columns=['Word', 'Frequency']
    )

    print(f"\n✓ WordCloud for abstracts generated.")
    print(f"--- Top {TOP_N} Most Frequent Words in Abstracts ---")
    print(df_abs_freq.to_string(index=False))

  PNG  → bibliometric_figures/wordcloud_abstracts.png

✓ WordCloud for abstracts generated.
--- Top 15 Most Frequent Words in Abstracts ---
         Word  Frequency
     learning        194
         data        127
        model        102
reinforcement         91
     proposed         89
    federated         85
      network         77
    framework         71
    algorithm         61
      privacy         59
          FRL         57
        local         55
  performance         54
       energy         54
         time         49


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 7 · SECTION B — Impact Analysis (Citations)
# ─────────────────────────────────────────────────────────────

if 'Citation Count' not in df.columns:
    print("⚠ Citation column not found. Verify if it is named 'Cited by' or 'TC'.")
else:
    # 1. Descriptive Statistics for PRISMA Discussion
    stats = df['Citation Count'].describe()

    print(f"--- Citation Impact Summary (N={int(stats['count'])}) ---")
    print(f"Total Citations in Dataset: {df['Citation Count'].sum()}")
    print(f"Average: {stats['mean']:.2f} | Median: {stats['50%']} | Max: {stats['max']}")
    print("-" * 50)

    # 2. Table of Top High-Impact Articles
    # Usamos COL_TITULO que definiste en las celdas anteriores
    df_top_impact = df[[COL_TITULO, 'Citation Count']].sort_values(by='Citation Count', ascending=False).head(TOP_N)

    # 3. Plotting Top Articles by Citation
    plt.figure(figsize=(12, 6))
    sns.barplot(
        x=df_top_impact['Citation Count'],
        y=df_top_impact[COL_TITULO].apply(lambda x: (x[:60] + '...') if len(x) > 60 else x),
        palette='viridis'
    )
    plt.title('Top Impact Articles by Citation Count', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Citations')
    plt.ylabel('Article Title')
    plt.tight_layout()

    # Guardar la figura
    guardar_fig_mpl(plt.gcf(), 'top_impact_articles')
    plt.show()

    print(f"\n--- List of Top {TOP_N} Articles ---")
    print(df_top_impact.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 7 · SECCIÓN B — Tendencias por año + regresión
# ─────────────────────────────────────────────────────────────

if not COL_AÑO:
    print('⚠ No se detectó columna de año.')
else:
    # Limpiar y extraer año numérico
    df['_año'] = (
        df[COL_AÑO].astype(str)
        .str.extract(r'(\d{4})', expand=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
    conteo = (df['_año'].dropna()
                        .astype(int)
                        .value_counts()
                        .sort_index()
                        .reset_index())
    conteo.columns = ['año', 'publicaciones']

    # Regresión lineal
    x = conteo['año'].values
    y = conteo['publicaciones'].values
    slope, intercept, r, p_val, se = stats.linregress(x, y)
    y_pred = slope * x + intercept
    tendencia = 'creciente 📈' if slope > 0 else 'decreciente 📉'
    sig = 'significativa (p<0.05)' if p_val < 0.05 else 'no significativa (p≥0.05)'

    # Gráfica interactiva Plotly
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=conteo['año'], y=conteo['publicaciones'],
        name='Publicaciones', marker_color='#4A90D9',
        hovertemplate='%{x}: %{y} artículos<extra></extra>'
    ))
    fig.add_trace(go.Scatter(
        x=conteo['año'], y=y_pred,
        mode='lines', name=f'Regresión (R²={r**2:.3f})',
        line=dict(color='#E8534A', width=2.5, dash='dash'),
        hovertemplate='%{x}: %{y:.1f} (tendencia)<extra></extra>'
    ))
    fig.update_layout(
        title=dict(text=f'Tendencia de publicaciones por año<br><sup>Tendencia {tendencia} — {sig} | pendiente={slope:.2f} art/año</sup>', font_size=16),
        xaxis_title='Año', yaxis_title='Número de publicaciones',
        xaxis=dict(tickmode='linear', dtick=1),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
        plot_bgcolor='white', paper_bgcolor='white',
        height=500,
    )
    fig.update_xaxes(showgrid=True, gridcolor='#eee')
    fig.update_yaxes(showgrid=True, gridcolor='#eee')
    fig.show()
    guardar_fig_plotly(fig, 'tendencias_año')

    print(f'\nResultados de regresión lineal:')
    print(f'  Tendencia   : {tendencia}')
    print(f'  Pendiente   : {slope:.4f} artículos/año')
    print(f'  R²          : {r**2:.4f}')
    print(f'  p-valor     : {p_val:.4f}  ({sig})')
    print(f'  Período     : {int(x.min())} – {int(x.max())}  ({len(x)} años)')
    print(f'  Total artic : {int(y.sum())}')

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




  PNG  → no disponible (kaleido); usa HTML
  HTML → bibliometric_figures/tendencias_año.html

Resultados de regresión lineal:
  Tendencia   : creciente 📈
  Pendiente   : 3.4000 artículos/año
  R²          : 0.4176
  p-valor     : 0.2387  (no significativa (p≥0.05))
  Período     : 2022 – 2026  (5 años)
  Total artic : 79


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 8 · SECCIÓN C — Top autores
# ─────────────────────────────────────────────────────────────

if not COL_AUTORES:
    print('⚠ No se detectó columna de autores.')
else:
    def separar_autores(texto):
        if not isinstance(texto, str):
            return []
        # Soporta separadores: ';', '|', ' and ', ','
        texto = re.sub(r'\band\b', ';', texto, flags=re.IGNORECASE)
        partes = re.split(r'[;|]', texto)
        return [p.strip() for p in partes if p.strip() and len(p.strip()) > 2]

    todos_autores = []
    for val in df[COL_AUTORES].dropna():
        todos_autores.extend(separar_autores(str(val)))

    conteo_aut = Counter(todos_autores).most_common(TOP_N)
    df_aut = pd.DataFrame(conteo_aut, columns=['autor', 'articulos'])

    fig = px.bar(
        df_aut.sort_values('articulos'),
        x='articulos', y='autor', orientation='h',
        title=f'Top {TOP_N} autores más productivos',
        labels={'articulos': 'Número de artículos', 'autor': ''},
        color='articulos',
        color_continuous_scale='Blues',
        text='articulos',
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(
        height=max(400, TOP_N * 32),
        showlegend=False,
        coloraxis_showscale=False,
        plot_bgcolor='white', paper_bgcolor='white',
        yaxis=dict(autorange='reversed'),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#eee')
    fig.show()
    guardar_fig_plotly(fig, 'top_autores')

    print(f'\n✓ {len(set(todos_autores))} autores únicos encontrados')
    print(f'   Top 5: {[(a, n) for a,n in conteo_aut[:5]]}')

  PNG  → no disponible (kaleido); usa HTML
  HTML → bibliometric_figures/top_autores.html

✓ 322 autores únicos encontrados
   Top 5: [('N. Kato', 4), ('L. Chen', 4), ('Y. Zhang', 4), ('Y. Liu', 3), ('Z. Li', 3)]


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 9 · SECCIÓN C — Top revistas / fuentes
# ─────────────────────────────────────────────────────────────

if not COL_REVISTA:
    print('⚠ No se detectó columna de revista/fuente.')
else:
    conteo_rev = (
        df[COL_REVISTA].dropna()
        .astype(str).str.strip().str.title()
        .value_counts()
        .head(TOP_N)
        .reset_index()
    )
    conteo_rev.columns = ['revista', 'articulos']

    fig = px.bar(
        conteo_rev.sort_values('articulos'),
        x='articulos', y='revista', orientation='h',
        title=f'Top {TOP_N} revistas / fuentes más frecuentes',
        labels={'articulos': 'Número de artículos', 'revista': ''},
        color='articulos',
        color_continuous_scale='Teal',
        text='articulos',
    )
    fig.update_traces(textposition='outside')
    fig.update_layout(
        height=max(400, TOP_N * 32),
        showlegend=False,
        coloraxis_showscale=False,
        plot_bgcolor='white', paper_bgcolor='white',
        yaxis=dict(autorange='reversed'),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#eee')
    fig.show()
    guardar_fig_plotly(fig, 'top_revistas')
    print(f'\n✓ {df[COL_REVISTA].nunique()} revistas únicas en el corpus')

  PNG  → no disponible (kaleido); usa HTML
  HTML → bibliometric_figures/top_revistas.html

✓ 0 revistas únicas en el corpus


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 10 · SECCIÓN D — Co-ocurrencia de keywords
# ─────────────────────────────────────────────────────────────

if not COL_KEYWORDS:
    print('⚠ No se detectó columna de keywords.')
    print('  Si tu archivo tiene keywords con otro nombre, asigna manualmente:')
    print('  COL_KEYWORDS = "nombre_de_tu_columna"')
else:
    def extraer_keywords(texto):
        if not isinstance(texto, str):
            return []
        partes = re.split(r'[;|,\n]', texto)
        return [p.strip().lower() for p in partes
                if p.strip() and len(p.strip()) > 2 and p.strip().lower() not in SW]

    # Frecuencia individual
    todas_kw = []
    listas_kw = []
    for val in df[COL_KEYWORDS].dropna():
        kws = extraer_keywords(str(val))
        todas_kw.extend(kws)
        if len(kws) >= 2:
            listas_kw.append(kws)

    freq_kw = Counter(todas_kw)
    top_kw  = freq_kw.most_common(TOP_N)

    # ── Barras: frecuencia de keywords ───────────────────────
    df_kw = pd.DataFrame(top_kw, columns=['keyword', 'frecuencia'])
    fig_kw = px.bar(
        df_kw.sort_values('frecuencia'),
        x='frecuencia', y='keyword', orientation='h',
        title=f'Top {TOP_N} keywords más frecuentes',
        color='frecuencia', color_continuous_scale='Purples',
        text='frecuencia',
    )
    fig_kw.update_traces(textposition='outside')
    fig_kw.update_layout(
        height=max(400, TOP_N * 32),
        showlegend=False, coloraxis_showscale=False,
        plot_bgcolor='white', paper_bgcolor='white',
        yaxis=dict(autorange='reversed'),
    )
    fig_kw.update_xaxes(showgrid=True, gridcolor='#eee')
    fig_kw.show()
    guardar_fig_plotly(fig_kw, 'frecuencia_keywords')

    # ── Co-ocurrencia: pares de keywords ─────────────────────
    pares = Counter()
    for lista in listas_kw:
        lista_unica = list(dict.fromkeys(lista))  # dedup manteniendo orden
        for a, b in combinations(lista_unica, 2):
            par = tuple(sorted([a, b]))
            pares[par] += 1

    top_pares = pares.most_common(TOP_COOC)
    df_pares  = pd.DataFrame(
        [(f'{a}  ·  {b}', n) for (a,b), n in top_pares],
        columns=['par', 'co_ocurrencias']
    )

    fig_cooc = px.bar(
        df_pares.sort_values('co_ocurrencias'),
        x='co_ocurrencias', y='par', orientation='h',
        title=f'Top {TOP_COOC} pares de keywords con mayor co-ocurrencia',
        color='co_ocurrencias', color_continuous_scale='Oranges',
        text='co_ocurrencias',
    )
    fig_cooc.update_traces(textposition='outside')
    fig_cooc.update_layout(
        height=max(500, TOP_COOC * 32),
        showlegend=False, coloraxis_showscale=False,
        plot_bgcolor='white', paper_bgcolor='white',
        yaxis=dict(autorange='reversed'),
    )
    fig_cooc.update_xaxes(showgrid=True, gridcolor='#eee')
    fig_cooc.show()
    guardar_fig_plotly(fig_cooc, 'coocurrencia_keywords')

    print(f'\n✓ {len(freq_kw)} keywords únicas encontradas')
    print(f'   {len(pares)} pares únicos de co-ocurrencia')

⚠ No se detectó columna de keywords.
  Si tu archivo tiene keywords con otro nombre, asigna manualmente:
  COL_KEYWORDS = "nombre_de_tu_columna"


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 11 · SECCIÓN E — Análisis automático de columnas
#  Genera gráficas para CUALQUIER columna categórica o numérica
#  que tenga datos suficientes y no haya sido analizada ya.
# ─────────────────────────────────────────────────────────────

# Columnas ya analizadas — se omiten aquí
ya_analizadas = set(filter(None, [
    COL_TITULO, COL_ABSTRACT, COL_AÑO, COL_AUTORES,
    COL_REVISTA, COL_KEYWORDS, '_año',
    'titulo_normalizado','fuente_archivo','formato',
    'decision_titulo','decision_abstract','decision_manual',
    'matches_titulo','matches_abstract',
    'score_titulo','score_abstract',
    'abstract_vacio','_duplicado_de',
]))

cols_restantes = [c for c in df.columns if c not in ya_analizadas]
figuras_auto = []

print(f'Analizando {len(cols_restantes)} columnas adicionales...\n')

for col in cols_restantes:
    serie = df[col].dropna()
    n_total   = len(df)
    n_llenos  = len(serie)
    cobertura = n_llenos / n_total

    if n_llenos < 3 or cobertura < 0.1:
        print(f'  ⏭  {col:40} ({n_llenos} valores, {cobertura:.0%}) — cobertura insuficiente')
        continue

    # Intentar numérica
    serie_num = pd.to_numeric(serie, errors='coerce').dropna()
    es_num    = len(serie_num) / n_llenos > 0.8

    if es_num and serie_num.nunique() > 4:
        # Histograma para columnas numéricas continuas (ej: citas, páginas, año)
        fig = px.histogram(
            x=serie_num, nbins=20,
            title=f'Distribución — {col}',
            labels={'x': col, 'count': 'Frecuencia'},
            color_discrete_sequence=['#6B8DD6'],
        )
        fig.update_layout(
            plot_bgcolor='white', paper_bgcolor='white', height=400,
            bargap=0.05,
        )
        fig.update_xaxes(showgrid=True, gridcolor='#eee')
        fig.update_yaxes(showgrid=True, gridcolor='#eee')
        fig.show()
        guardar_fig_plotly(fig, f'auto_{col[:30]}')
        figuras_auto.append(col)
        print(f'  ✓  {col:40} → histograma (media={serie_num.mean():.1f}, σ={serie_num.std():.1f})')

    else:
        # Barras para columnas categóricas
        # Separar si los valores contienen ' | ' (campos multi-valor)
        if serie.astype(str).str.contains(r'\|').any():
            valores = []
            for v in serie.astype(str):
                valores.extend([x.strip() for x in v.split('|') if x.strip()])
            conteo = Counter(valores)
        else:
            conteo = Counter(serie.astype(str).str.strip().str.title().tolist())

        n_unicos = len(conteo)
        if n_unicos < 2 or n_unicos > 200:
            print(f'  ⏭  {col:40} ({n_unicos} valores únicos) — demasiados o muy pocos')
            continue

        top = conteo.most_common(TOP_N)
        df_top = pd.DataFrame(top, columns=['valor', 'n'])

        fig = px.bar(
            df_top.sort_values('n'),
            x='n', y='valor', orientation='h',
            title=f'Top valores — {col}  ({n_unicos} únicos)',
            labels={'n': 'Frecuencia', 'valor': ''},
            color='n', color_continuous_scale='Teal',
            text='n',
        )
        fig.update_traces(textposition='outside')
        fig.update_layout(
            height=max(350, min(TOP_N, len(top)) * 30),
            showlegend=False, coloraxis_showscale=False,
            plot_bgcolor='white', paper_bgcolor='white',
            yaxis=dict(autorange='reversed'),
        )
        fig.update_xaxes(showgrid=True, gridcolor='#eee')
        fig.show()
        guardar_fig_plotly(fig, f'auto_{col[:30]}')
        figuras_auto.append(col)
        print(f'  ✓  {col:40} → barras ({n_unicos} categorías únicas)')

print(f'\n✓ {len(figuras_auto)} gráficas adicionales generadas automáticamente')

Analizando 1 columnas adicionales...



  PNG  → no disponible (kaleido); usa HTML
  HTML → bibliometric_figures/auto_doi.html
  ✓  doi                                      → barras (79 categorías únicas)

✓ 1 gráficas adicionales generadas automáticamente


In [ ]:
# ─────────────────────────────────────────────────────────────
#  CELDA 12 · Empaquetar y descargar todas las figuras
# ─────────────────────────────────────────────────────────────
import zipfile, glob
from google.colab import files

ZIP_NOMBRE = 'figuras_bibliometria.zip'

archivos = glob.glob(f'{CARPETA_PNG}/**/*', recursive=True)
archivos = [a for a in archivos if os.path.isfile(a)]

with zipfile.ZipFile(ZIP_NOMBRE, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arch in archivos:
        zf.write(arch)

print(f'ZIP con {len(archivos)} archivos:')
for a in sorted(archivos):
    size = os.path.getsize(a)
    print(f'  {a:55} {size/1024:.0f} KB')

print(f'\nDescargando {ZIP_NOMBRE}...')
files.download(ZIP_NOMBRE)
print('\n✓ Análisis bibliométrico completo.')

ZIP con 7 archivos:
  bibliometric_figures/auto_doi.html                      4461 KB
  bibliometric_figures/tendencias_año.html                4461 KB
  bibliometric_figures/top_autores.html                   4461 KB
  bibliometric_figures/top_revistas.html                  4461 KB
  bibliometric_figures/wordcloud_abstracts.png            666 KB
  bibliometric_figures/wordcloud_titles.png               558 KB
  bibliometric_figures/wordcloud_titulos.png              505 KB

Descargando figuras_bibliometria.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Análisis bibliométrico completo.
